In [2]:
import itertools
import concurrent.futures
import constants as Cs
import os
import json
import datetime
import pickle
import optuna
import numpy as np
from scipy.spatial.distance import pdist
import logging


SEEDS = [101,102,103]
TEST_EVAL_EPS = 5
# lambda fit archving [FrozenTrial(number=80, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 4, 53, 822616), datetime_complete=datetime.datetime(2026, 5, 22, 5, 36, 8, 359073), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=81, value=None), FrozenTrial(number=86, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 36, 31, 908497), datetime_complete=datetime.datetime(2026, 5, 22, 6, 11, 5, 128793), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=87, value=None)]
# lambda - this one is bad actually novelty [FrozenTrial(number=4, state=<TrialState.COMPLETE: 1>, values=[-9.291787437438707], datetime_start=datetime.datetime(2026, 5, 19, 22, 25, 13, 886515), datetime_complete=datetime.datetime(2026, 5, 19, 23, 0, 16, 677521), params={'crossmethod': 'mean', 'lambda': 60, 'mu': 60, 'mutation_rate': 0.18, 'cross_rate': 1.0, 'sigma': 2.5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5)}, trial_id=208, value=None)]
EXAMPLE_MAPPING  = {
    "lambda":"l",
    "mu": "m",
    "mutation_rate":"mr",
    "cross_rate":"cr",
    "sigma": "mutation_sigma",
    "cross":"cross_uni",
    "crossmethod":"cross_method"
}

def rename(dict):
    return {EXAMPLE_MAPPING.get(k, k): v for k, v in dict.items()}

def task_job(en, alg, container, args, s, out_path):
    env = Cs.ENIVROMENTS[en]()
    df, pop = Cs.ALG_MAPPING[alg].argumented_function(env=en, container=container, seed=s, out_path=out_path, **args)
    print("Testing " + str(s))
    fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for p in pop]
    print("Finished seed %d of algorithm %s" % (s, alg))
    return fitnesses, pop


def evaluation_of_setup(en, run_name, alg, container, out_path, ev_ng, **kwargs):
    #we do not deviate the sigma
    # first we evaluate the current setup

    stat_futures = {}
    args = rename(kwargs)
    args["ng"] = ev_ng
    dirpath = os.path.join(os.path.realpath(out_path), container,alg, run_name)

    with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
        print("Launching " + alg + "on Enviroment " + str(en) + f" with {args}")
        for s in SEEDS:
            future = executor.submit(task_job, container=container, alg=alg, en=en, args=args, s=s, out_path=dirpath + "/stat")
            stat_futures[future] = s

    stats = {}
    pops = {}
    maxes = []
    diversities = []
    for future in concurrent.futures.as_completed(stat_futures):
        s = stat_futures[future]
        result = future.result()
        if "fitness" not in stats:
            stats["fitness"] = {}
        stats["fitness"][s] = result[0]
        behaviors = list(map(lambda x:x[1], result[0]))
        fitnesses = list(map(lambda x:x[0], result[0]))
        maxes.append(np.max(fitnesses))
        diversity = pdist(np.array(behaviors)).mean()
        diversities.append(diversity)
        pops[s]= result[1] 
    ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"{ts}_{container}_"+ "|".join(f"{k}: {v}" for k, v in args.items())
    stats["environment"] = en
    stats["algorithm"] = alg
    stats["container"] = container
    stats["arguments"] = args
    json_path = os.path.join(dirpath, filename +".json")

    with open(json_path, "w") as json_file:
        json.dump(stats,json_file)
        print("Finished "+ filename)
    return np.mean(maxes), np.mean(diversity)
  



2026-06-02 22:33:22.096364: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-02 22:33:22.990081: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

from concurrent.futures import Future
def select_minimal_exaples(examples):
    pop = np.inf
    selected_trials = []
    selected_trials_index = []
    for i, t in enumerate(examples):
        if "lambda" in t:
            k = t["lambda"]
            if "mu" in t:
                k+=t["mu"]
        elif "pop" in t:
            k=t["pop"]
        else: raise NameError(f"wtf")
        if pop == k:
            selected_trials.append(t)
        elif pop > k:
            pop = k
            selected_trials = [t]
            selected_trials_index.append(i)

    return selected_trials, selected_trials_index

def argument_combination_genration(selected, D_args):
    deltas = []
    contraints_global = []
    for i, a in enumerate(D_args):
        delta = D_args[a][0]
        constraints = D_args[a][1]
        deltas.append(delta)
        contraints_global.append(constraints)

    generated_arguments = []

    for delta in itertools.product(*deltas):
        new = selected.copy()
        keep = True
        for i, a in enumerate(D_args):
            new[a] = selected[a] + delta[i]
            passed = (len(contraints_global[i]) == 0 or
            contraints_global[i][0] is None or
            contraints_global[i][0] < new[a] or
            contraints_global[i][1] is None or
            contraints_global[i][1] > new[a])
            keep = keep and passed

        if not keep: continue
    

        generated_arguments.append(new)
    return generated_arguments

def process_generated_arguments(run_name, en, container, alg, generated_arguments, selected_fitnes, selected_diversity, visited, outpath):
    visited_new = visited.copy()
    with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
        arg_futures={}        
        for args in generated_arguments:
            toupled = tuple(sorted(args.items()))
            if toupled in visited:
                future = Future().set_result(visited[toupled])

            future = executor.submit(
                evaluation_of_setup, 
                run_name=run_name,
                en=en, 
                alg=alg,
                ev_ng=20, 
                container=container,
                out_path=outpath,
                **args)
            arg_futures[future] = args
        max_fitness = selected_fitnes
        max_diversity = selected_diversity
        return_candidate = None
        for future in concurrent.futures.as_completed(arg_futures):
            args = arg_futures[future]
            toupled = tuple(sorted(args.items()))
            fitness, diversity  = future.result()
            visited_new[toupled] = (fitness, diversity)
            if fitness > max_fitness:
                print("We should have changed who's the best!")
                max_fitness = fitness
                return_candidate = args
                max_diversity = diversity
            elif fitness == max_fitness:
                if diversity > max_diversity:
                    print("We should have changed based on diversity!")
                    max_fitness = fitness
                    return_candidate = args
                    max_diversity = diversity
                
        return max_fitness, max_diversity, return_candidate, visited_new

            


In [4]:

def adaptive_lambda_grid_search(run_name, en,container, hops, starting_position, starting_fitness, dl, dm, dcr, dmr, outpath):
    visited = dict()
    visited[tuple(sorted(starting_position.items()))] = (starting_fitness, 0)
    selected = starting_position
    selected_fitness = starting_fitness
    selected_diversity = 0
    D_cr = [dcr, 0, -dcr]
    D_mr = [dmr, 0, -dmr]
    D_m = [dm, 0, -dm]
    D_l = [dl, 0, -dl]
    one_success = False
    for i in range(hops):

        generated_arguments = argument_combination_genration(
            selected=selected, 
            D_args={"cr":[D_cr, (0,1)], "l":[D_l, (0,None)]})
        # we have finised our 
        if generated_arguments == []: return selected, selected_fitness

        selection_fitness, selection_diversity, new_selected, new_visited = process_generated_arguments(
            run_name=run_name,
            en=en, 
            container=container, 
            alg="lambda",
            visited=visited, 
            generated_arguments=generated_arguments, 
            selected_fitnes=selected_fitness,
            selected_diversity=selected_diversity,
            outpath=outpath
        )
        if new_selected is not None:
            one_success = True
            if tuple(sorted(new_selected.items())) in visited:
                print("Cycle found! Terminating")
                return selected, new_visited
            selected = new_selected
            selected_diversity = selection_diversity
            selected_fitness = selection_fitness
        visited = new_visited
        generated_arguments = argument_combination_genration(
            selected=selected, 
            D_args={"mr":[D_mr, (0,1)], "m":[D_m, (0,None)]})
        # we have finised our 
        if generated_arguments == []: return selected, selected_fitness
        selection_fitness, selection_diversity, new_selected, new_visited = process_generated_arguments(
            run_name=run_name,
            en=en, 
            container=container, 
            alg="lambda", 
            generated_arguments=generated_arguments, 
            selected_fitnes=selected_fitness,
            selected_diversity=selected_diversity,
            visited=visited,
            outpath=outpath
        )
        if new_selected is None:
            visited = new_visited
        else:
            one_success = True
            if tuple(sorted(new_selected.items())) in visited:
                print("Cycle found! Terminating")
                return selected, new_visited
            visited = new_visited
            selected = new_selected
            selected_diversity = selection_diversity
            selected_fitness = selection_fitness
        
        if not one_success:
            return selected, visited
        
    
    



In [5]:

def adaptive_diff_grid_search(
        run_name,
        en,
        container, 
        hops, 
        starting_position, 
        starting_fitness, 
        dl, 
        dcr, 
        dmr, 
        outpath
    ):
    visited = dict()
    visited[tuple(sorted(starting_position.items()))] = (starting_fitness, 0)
    selected = starting_position
    selected_fitness = starting_fitness
    selected_diversity = 0
    D_cr = [dcr, 0, -dcr]
    D_mr = [dmr, 0, -dmr]
    D_l = [dl, 0, -dl]
    one_success = False
    for i in range(hops):

        generated_arguments = argument_combination_genration(
            selected=selected, 
            D_args={"l":[D_l, (0,None)]})
        # we have finised our 
        if generated_arguments == []: return selected, selected_fitness

        selection_fitness, selection_diversity, new_selected, new_visited = process_generated_arguments(
            run_name=run_name,
            en=en, 
            container=container, 
            alg="diff",
            visited=visited, 
            generated_arguments=generated_arguments, 
            selected_fitnes=selected_fitness,
            selected_diversity=selected_diversity,
            outpath=outpath
        )
        if new_selected is not None:
            one_success = True
            if tuple(sorted(new_selected.items())) in visited:
                print("Cycle found! Terminating")
                return selected, new_visited
            selected = new_selected
            selected_diversity = selection_diversity
            selected_fitness = selection_fitness
        visited = new_visited


        generated_arguments = argument_combination_genration(
            selected=selected, 
            D_args={"mr":[D_mr, (0,1)], "cr":[D_cr, (0,None)]})
        # we have finised our 
        if generated_arguments == []: return selected, selected_fitness
        selection_fitness, selection_diversity, new_selected, new_visited = process_generated_arguments(
            run_name=run_name,
            en=en, 
            container=container, 
            alg="diff",
            visited=visited, 
            generated_arguments=generated_arguments, 
            selected_fitnes=selected_fitness,
            selected_diversity=selected_diversity,
            outpath=outpath
        )
        if new_selected is None:
            visited = new_visited
        else:
            one_success = True
            if tuple(sorted(new_selected.items())) in visited:
                print("Cycle found! Terminating")
                return selected, new_visited
            visited = new_visited
            selected = new_selected
            selected_diversity = selection_diversity
            selected_fitness = selection_fitness
        
        if not one_success:
            return selected, visited
        
    
    



In [6]:
def adaptive_grid_search(en, alg, run_name, container, hops = 3, out_path="./Data/grid_search"):
    with open("relevant_studies.json", "r") as f:
        relevant_study_names = json.load(f)
    storage = f"sqlite:///Data/optuna/{en}/{container}/{alg}.db"
    study_name = relevant_study_names[container][alg][en]

    new_study = optuna.load_study(study_name=study_name,storage=storage)
    most_promising, mi = select_minimal_exaples([t.params for t in new_study.best_trials])
    selected_trial = new_study.best_trials[mi[0]]
    start =datetime.datetime.now()
    
    if alg=="lambda":
        if en == "lunarlander":
            dl = 15
            dm = 15
            dcr = 0.1
            dmr = 0.05
        else: #cartpole
            dl = 10
            dm = 10
            dcr = 0.1
            dmr = 0.05
        selected, visited = adaptive_lambda_grid_search(
            run_name=run_name,
            en=en,
            container=container, 
            hops=hops, 
            starting_position=rename(most_promising[0]),
            starting_fitness=selected_trial.value, 
            dl=dl, 
            dm=dm, 
            dcr=dcr, 
            dmr=dmr,
            outpath=out_path+f"/{en}"
        )
    elif alg =="diff":
        if en == "lunarlander":
            dl = 10
            dcr = 0.1
            dmr = 0.1
        else: #cartpole
            dl = 5
            dcr = 0.1
            dmr = 0.1
        selected, visited = adaptive_diff_grid_search(
            run_name=run_name,
            en=en,
            container=container, 
            hops=hops, 
            starting_position=rename(most_promising[0]),
            starting_fitness=selected_trial.value, 
            dl=dl,  
            dcr=dcr, 
            dmr=dmr,
            outpath=out_path+f"/{en}"
        )
    end =datetime.datetime.now()
    ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"{ts}_{container}_{en}_protocol.json"
    dirpath = os.path.join(os.path.realpath(out_path),en, container,alg)

    protocol = {"start":start.strftime("%Y-%m-%d_%H-%M-%S"), "end": end.strftime("%Y-%m-%d_%H-%M-%S"), "origin": most_promising, "final":selected, "visited":[dict(k) for k in visited]}
    json_path = os.path.join(dirpath, filename + ".json")
    with open(json_path, "w") as json_file:
        json.dump(protocol,json_file)


    



In [7]:
protocol = {}
#json_path = os.path.join(dirpath, filename + ".json")
json_path = "/home/schkliba/git/MastersThesis/Data/grid_search/first_try/lunarlander/novelty/lambda/2026-06-02_07-15-37_novelty_cross_method: uniform|l: 70|m: 40|mr: 0.06|cr: 0.7|mutation_sigma: 2.5|cross_uni: 0.2|ng: 20.json"
with open(json_path, "w") as json_file:
        json.dump(protocol,json_file)

In [10]:
adaptive_grid_search(
    en="lunarlander", 
    alg="lambda", 
    container="fitness", run_name="fitness", hops = 3, out_path="./Data/grid_search/first_try")


Launching lambdaon Enviroment lunarlander with {'cross_method': 'uniform', 'l': 50, 'm': 50, 'mr': 0.03, 'cr': 0.9, 'ng': 20}Launching lambdaon Enviroment lunarlander with {'cross_method': 'uniform', 'l': 65, 'm': 50, 'mr': 0.03, 'cr': 0.9, 'ng': 20}Launching lambdaon Enviroment lunarlander with {'cross_method': 'uniform', 'l': 65, 'm': 50, 'mr': 0.03, 'cr': 0.8, 'ng': 20}

Launching lambdaon Enviroment lunarlander with {'cross_method': 'uniform', 'l': 35, 'm': 50, 'mr': 0.03, 'cr': 0.9, 'ng': 20}

Launching lambdaon Enviroment lunarlander with {'cross_method': 'uniform', 'l': 50, 'm': 50, 'mr': 0.03, 'cr': 0.8, 'ng': 20}


2026-06-03 06:58:08.844304: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-03 06:58:08.853509: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2026-06-03 06:58:09.288113: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/li

gen	nevals	avg     	min     	max     	std    
0  	35    	-405.677	-602.368	-125.803	130.429
gen	nevals	avg     	min     	max    	std   
0  	35    	-423.385	-713.035	-100.56	176.39
gen	nevals	avg     	min     	max     	std    
0  	35    	-409.813	-682.896	-67.5599	186.486
gen	nevals	avg     	min     	max     	std    
0  	50    	-376.934	-616.413	-125.803	145.317
gen	nevals	avg     	min     	max     	std    
0  	50    	-376.934	-616.413	-125.803	145.317
gen	nevals	avg     	min     	max    	std    
0  	50    	-442.715	-731.976	-100.56	174.705
gen	nevals	avg     	min     	max    	std    
0  	50    	-442.715	-731.976	-100.56	174.705
gen	nevals	avg     	min     	max     	std    
0  	50    	-430.693	-682.896	-67.5599	185.759
gen	nevals	avg     	min     	max     	std    
0  	50    	-430.693	-682.896	-67.5599	185.759
gen	nevals	avg     	min     	max     	std    
0  	65    	-384.232	-616.413	-125.803	143.899
gen	nevals	avg     	min     	max     	std    
0  	65    	-384.232	-616.413	-125.803	143.

2026-06-03 07:57:28.753061: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-03 07:57:28.765788: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


18 	50    	-32.9536	-32.9536	-32.9536	7.10543e-15
18 	46    	-67.5599	-67.5599	-67.5599	0      
16 	35    	101.587 	101.587 	101.587 	1.42109e-14
17 	47    	3.81687 	3.81687 	3.81687 	4.44089e-16
19 	41    	-25.7305	-25.7305	-25.7305	0      
18 	42    	97.2136 	97.2136 	97.2136 	1.42109e-14
17 	46    	23.5739 	-63.3737	66.9194 	31.3454
18 	34    	3.81687 	3.81687 	3.81687 	4.44089e-16
18 	45    	3.82617 	-77.215 	49.7201 	30.9715
18 	38    	32.1942   	-162.015	82.9024 	36.9303
19 	46    	-39.4808	-39.4808	-39.4808	0      
19 	48    	-67.5599	-67.5599	-67.5599	0      
19 	47    	14.2012 	14.2012 	14.2012 	1.77636e-15
19 	46    	-32.9536	-32.9536	-32.9536	7.10543e-15
17 	40    	101.587 	101.587 	101.587 	1.42109e-14
18 	46    	3.81687 	3.81687 	3.81687 	4.44089e-16
20 	41    	-25.7305	-25.7305	-25.7305	0      
Testing 103
19 	35    	97.2136 	97.2136 	97.2136 	1.42109e-14
19 	39    	3.81687 	3.81687 	3.81687 	4.44089e-16
18 	46    	24.8752 	-258.263	66.9194 	61.4744
Finished seed 103 of a

...

14 	25    	-54.1678	-54.1678	-54.1678	7.10543e-15
20 	53    	12.0855 	12.0855 	12.0855 	3.55271e-15
Testing 103
13 	25    	-56.0702	-65.5512	12.0348 	25.1495
1  	53    	-241.767	-490.694	-99.1719	114.899
11 	30    	-18.1304	-24.6437	9.5906  	8.26568
18 	46    	60.421  	24.5552 	93.2219 	24.3636
1  	50    	-262.61 	-567.515	-70.1584	145.162
20 	53    	-16.9925	-106.606	5.01325 	16.9108
Testing 101
gen	nevals	avg     	min     	max     	std    
0  	50    	-376.934	-616.413	-125.803	145.317
gen	nevals	avg     	min     	max    	std    
0  	50    	-442.715	-731.976	-100.56	174.705
1  	51    	-282.21 	-620.329	-67.5599	136.325
15 	32    	-54.1678	-54.1678	-54.1678	7.10543e-15
gen	nevals	avg     	min     	max     	std    
0  	50    	-430.693	-682.896	-67.5599	185.759
14 	31    	-48.3275	-65.3528	12.0348 	32.0575
18 	45    	60.4548 	60.4548 	60.4548 	0      
12 	29    	-15.4687	-32.8008	9.5906  	9.46252
2  	50    	-147.457	-317.495	-49.4333	69.6694
Finished seed 101 of algorithm lambda


2026-06-03 10:32:00.082010: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-03 10:32:00.095115: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


16 	28    	-54.1678	-54.1678	-54.1678	7.10543e-15
1  	38    	-260.509	-572.615	-70.1584	140.021
1  	43    	-212.808	-490.694	-99.1719	110.568
2  	47    	-181.519	-417.654	-70.1584	103.575
1  	40    	-286.719	-616.754	-67.5599	151.099
15 	31    	-23.5635	-65.3528	12.0348 	38.5698
19 	42    	76.7257 	24.5552 	93.2219 	15.8598
2  	54    	-195.746	-505.132	-33.256 	115.025
13 	28    	-6.73687	-24.6437	32.3377 	14.5406
17 	34    	-54.1678	-54.1678	-54.1678	7.10543e-15
2  	36    	-182.934	-489.923	-70.1584	108.564
2  	41    	-148.844	-268.99 	-101.859	41.5512
Finished seed 103 of algorithm lambda
Finished seed 101 of algorithm lambda
3  	53    	-108.534	-232.772	-49.4333	47.712 
16 	29    	0.863824	-65.3528	12.0348 	26.3761
3  	55    	-127.792	-344.195	-70.1584	71.0816
2  	43    	-162.277	-545.782	-67.5599	90.9447
14 	27    	-2.58293	-15.9597	32.3377 	10.3486
18 	31    	-54.1678	-54.1678	-54.1678	7.10543e-15
19 	56    	60.4548 	60.4548 	60.4548 	0      
3  	39    	-164.544	-523.548	-70.1584	

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/

6  	44    	-79.5511	-161.117	-65.3949	17.1675
5  	52    	-85.3821	-261.554	-33.256 	52.4737
18 	29    	18.1675 	-10.5918	45.9581 	14.1105
Finished seed 103 of algorithm lambda
7  	41    	-88.5051	-180.243	97.4696 	40.2903
6  	56    	-94.9092	-359.159	-58.5234	53.7516
7  	44    	-55.9407	-293.813	-13.9252	38.7515
6  	41    	-14.8841	-91.341 	21.7414 	47.6354
7  	41    	-85.7247	-503.905	-38.4647	74.9358
gen	nevals	avg     	min     	max    	std    
0  	50    	-442.715	-731.976	-100.56	174.705
gen	nevals	avg     	min     	max     	std    
0  	50    	-376.934	-616.413	-125.803	145.317
19 	27    	25.42   	-41.506 	83.3085 	23.8683
gen	nevals	avg     	min     	max     	std    
0  	50    	-430.693	-682.896	-67.5599	185.759
Finished seed 102 of algorithm lambda
Finished 2026-06-03_10-46-21_fitness_cross_method: uniform|l: 50|m: 65|mr: 0.08|cr: 0.8|ng: 20
8  	38    	-71.6178	-101.859	97.4696 	56.4537
6  	54    	-65.3777	-335.078	-33.256 	50.505 
7  	51    	-86.2039	-269.393	-58.238 	28.993 
1  